In [1]:
print("all ok")


all ok


Tool calling means an LLM can decide to use an external function/tool when it needs information or wants to perform an action, instead of answering only from its own knowledge.

Tool calling is the ability of an LLM to select and invoke external tools or functions to get information or perform actions beyond its built-in knowledge.

## these are some of the example 
### but the concept is we can convert any of the funcationality into the tool

Web search
Calculator
Database query
API calls
retriever
Email sending
Weather API
Stock-price API
File operations

User: "What is the weather in Bangalore today?"

        ↓

LLM decides: "I need current weather data."

        ↓

Calls Weather Tool

        ↓

Tool returns: 28°C, cloudy

        ↓

LLM gives final answer

Node = graph decides when to execute the function.
Tool = LLM decides when to execute the function.

User
 ↓
LLM
 ↓
Does it need a tool?
 ↓
Yes
 ↓
Tool(function) Call
 ↓
Tool Result
 ↓
LLM
 ↓
Final Answer

MCP Server
   │
   ├── Tool 1
   ├── Tool 2
   └── Tool 3
        ↓
langchain-mcp-adapters
        ↓
LangChain BaseTools
        ↓
LangGraph Agent

Now if the user asks: "What is 20 + 30?"

the LLM may produce a tool request like:

Tool: add
Arguments:
a = 20
b = 30

Then the tool executes: 50

and the LLM can use that result to answer.

The LLM does not execute the function itself. It decides which tool(function) to call and with what arguments; your application executes the tool and returns the result to the LLM.

LLM = decision maker
Tool = actual executor(acutal funcationality)

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

print("Setup loaded.")
print("GROQ_API_KEY available:", bool(os.getenv("GROQ_API_KEY")))
print("OPENAI_API_KEY available:", bool(os.getenv("OPENAI_API_KEY")))
print("TAVILY_API_KEY available:", bool(os.getenv("TAVILY_API_KEY")))

Setup loaded.
GROQ_API_KEY available: True
OPENAI_API_KEY available: True
TAVILY_API_KEY available: True


## 1. `@tool` decorator

In [3]:
from langchain_core.tools import tool

In [4]:
@tool
def add_basic(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [5]:
print("Name:", add_basic.name)

Name: add_basic


In [6]:
print("Description:", add_basic.description)

Description: Add two numbers.


In [7]:
print("Args:", add_basic.args)

Args: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [8]:
add_basic.invoke({"a": 20, "b": 30})

50

In [9]:
result = add_basic.invoke({"a": 20, "b": 30})
print("Execution result:", result)

Execution result: 50


## 2. `@tool("custom_name")`

In [10]:
@tool("calculator")
def add_with_custom_name(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [11]:
print("Tool name:", add_with_custom_name.name)
print("Execution result:", add_with_custom_name.invoke({"a": 10, "b": 15}))

Tool name: calculator
Execution result: 25


In [23]:
@tool(
    "multiply_numbers",
    description="Multiply two integers and return the result.",
    return_direct=False,
)
def multiply_with_options(a, b):
    return a * b

In [24]:
print("Execution result:", multiply_with_options.invoke({"a": "sunny", "b": 7}))

Execution result: sunnysunnysunnysunnysunnysunnysunny


In [14]:
print("Name:", multiply_with_options.name)
print("Description:", multiply_with_options.description)
print("return_direct:", multiply_with_options.return_direct)
print("Execution result:", multiply_with_options.invoke({"a": 6, "b": 7}))

Name: multiply_numbers
Description: Multiply two integers and return the result.
return_direct: False
Execution result: 42


## 4. `@tool` with Pydantic

In [15]:
from pydantic import BaseModel, Field, ValidationError

In [16]:
class CalculatorInputTest(BaseModel):
    a: int = Field(description="First integer")
    b: int = Field(description="Second integer")

In [17]:
@tool(args_schema=CalculatorInputTest)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [18]:
print("Schema:", multiply.args_schema.model_json_schema())

Schema: {'properties': {'a': {'description': 'First integer', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second integer', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'CalculatorInputTest', 'type': 'object'}


In [19]:
print("Execution result:", multiply.invoke({"a": 8, "b": 9}))

Execution result: 72


In [20]:
multiply.invoke({"a": "sunny", "b": 9})

ValidationError: 1 validation error for CalculatorInputTest
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='sunny', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

## 5. `@tool(parse_docstring=True)`

In [25]:
@tool(parse_docstring=True)
def search_with_docstring(query: str, limit: int) -> str:
    """Search documents.

    Args:
        query: Search query entered by the user.
        limit: Maximum number of results.
    """
    return f"Searching for '{query}' with limit={limit}"

In [26]:
print("Args schema:")
print(search_with_docstring.args_schema.model_json_schema())

Args schema:
{'description': 'Search documents.', 'properties': {'query': {'description': 'Search query entered by the user.', 'title': 'Query', 'type': 'string'}, 'limit': {'description': 'Maximum number of results.', 'title': 'Limit', 'type': 'integer'}}, 'required': ['query', 'limit'], 'title': 'search_with_docstring', 'type': 'object'}


In [27]:
print("Execution result:", search_with_docstring.invoke({"query": "LangGraph memory","limit": 3,}))


Execution result: Searching for 'LangGraph memory' with limit=3


## 6. Async function with `@tool`

In [28]:
import asyncio

In [30]:
@tool
async def get_data_async(url: str) -> str:
    """Fetch data asynchronously."""
    await asyncio.sleep(0.1)
    return f"Data from {url}"

In [31]:
result = await get_data_async.ainvoke({"url": "https://example.com"})
print("Async execution result:", result)

Async execution result: Data from https://example.com


## 7. Tool with `ToolRuntime` 

<!-- from typing_extensions import TypedDict
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, ToolRuntime
from langchain_core.messages import AIMessage

class RuntimeState(MessagesState):
    question: str

@tool
def read_question_from_runtime(runtime: ToolRuntime) -> str:
    """Read the current question from LangGraph state."""
    return runtime.state["question"]

def create_demo_tool_call(state: RuntimeState):
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "read_question_from_runtime",
                    "args": {},
                    "id": "demo_call_1",
                    "type": "tool_call",
                }],
            )
        ]
    }

runtime_builder = StateGraph(RuntimeState)
runtime_builder.add_node("create_call", create_demo_tool_call)
runtime_builder.add_node("tools", ToolNode([read_question_from_runtime]))
runtime_builder.add_edge(START, "create_call")
runtime_builder.add_edge("create_call", "tools")
runtime_builder.add_edge("tools", END)

runtime_graph = runtime_builder.compile()

runtime_result = runtime_graph.invoke({
    "question": "What is LangGraph?",
    "messages": [],
})

print("Tool result:", runtime_result["messages"][-1].content) -->

## 8. `Tool(...)` constructor"

In [32]:
from langchain_core.tools import Tool

In [33]:
def simple_search_function(query: str) -> str:
    return f"Searching for {query}"

In [35]:
simple_search_tool = Tool(
    name="simple_search",
    func=simple_search_function,
    description="Search for information.",
)

In [36]:
print("Name:", simple_search_tool.name)
print("Execution result:", simple_search_tool.invoke("LangGraph"))

Name: simple_search
Execution result: Searching for LangGraph


In [37]:
def search_from_function(query: str) -> str:
    return f"Result for {query}"

In [38]:
Tool.from_function(
    func=search_from_function,
    name="search_from_function",
    description="Search information.",
)

Tool(name='search_from_function', description='Search information.', func=<function search_from_function at 0x000001269F8498A0>)

In [39]:
from_function_tool = Tool.from_function(
    func=search_from_function,
    name="search_from_function",
    description="Search information.",
)

In [40]:
print("Execution result:", from_function_tool.invoke("Agentic AI"))

Execution result: Result for Agentic AI


## 10. `StructuredTool.from_function()`

In [41]:
from langchain_core.tools import StructuredTool

def calculate_tax_test(income: float, tax_rate: float) -> float:
    return income * tax_rate

tax_tool_test = StructuredTool.from_function(
    func=calculate_tax_test,
    name="calculate_tax",
    description="Calculate tax from income and tax rate.",
)

print("Args:", tax_tool_test.args)
print("Execution result:", tax_tool_test.invoke({
    "income": 100000,
    "tax_rate": 0.20,
}))

Args: {'income': {'title': 'Income', 'type': 'number'}, 'tax_rate': {'title': 'Tax Rate', 'type': 'number'}}
Execution result: 20000.0


In [43]:
class MultiplyInputTest2(BaseModel):
    a: int = Field(description="First number")
    b: int = Field(description="Second number")
class MultiplyInputTest2(BaseModel):
    a: int = Field(description="First number")
    b: int = Field(description="Second number")

def direct_multiply(a: int, b: int) -> int:
    return a * b

direct_structured_tool = StructuredTool(
    name="direct_multiply",
    description="Multiply two numbers.",
    func=direct_multiply,
    args_schema=MultiplyInputTest2,
)

print("Execution result:", direct_structured_tool.invoke({
    "a": 12,
    "b": 4,
}))

Execution result: 48


## 12. Subclass `BaseTool`

In [44]:
from typing import Type
from langchain_core.tools import BaseTool

In [45]:
class SearchInputTest(BaseModel):
    query: str = Field(description="Search query")

In [46]:
class MySearchToolTest(BaseTool):
    name: str = "my_search"
    description: str = "Search my custom database."
    args_schema: Type[BaseModel] = SearchInputTest

    def _run(self, query: str) -> str:
        return f"Custom database result for: {query}"

In [47]:
custom_base_tool = MySearchToolTest()

In [48]:
print("Execution result:", custom_base_tool.invoke({
    "query": "LangGraph state management"
}))

Execution result: Custom database result for: LangGraph state management


## 13. `create_retriever_tool()`

In [ ]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.tools import create_retriever_tool

In [ ]:
class DemoRetriever(BaseRetriever):
    def _get_relevant_documents(self, query: str, *, run_manager=None):
        return [
            Document(
                page_content=f"Demo internal document relevant to: {query}",
                metadata={"source": "demo.txt"},
            )
        ]

demo_retriever = DemoRetriever()

In [ ]:
retriever_tool_test = create_retriever_tool(
    demo_retriever,
    name="search_company_documents",
    description="Search internal company documents.",
)

In [ ]:
print("Execution result:")
print(retriever_tool_test.invoke({
    "query": "What is the company leave policy?"
}))

## Model setup for schema/binding tests

In [60]:
groq_llm = None

In [61]:
if os.getenv("GROQ_API_KEY"):
    from langchain_groq import ChatGroq

    groq_llm = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0,
    )
    print("Groq model initialized.")
else:
    print("Skipping live Groq tests because GROQ_API_KEY is not available.")

Groq model initialized.


In [62]:
groq_llm.invoke("hi")

[08/30/26 20:36:57] INFO     HTTP Request: POST https://api.groq.com/openai/v1/chat/completions     ]8;id=572951;file://d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=572952;file://d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond. The instruction: "You are ChatGPT, a large language model trained by OpenAI." There\'s no special instruction. Just respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 72, 'total_tokens': 129, 'completion_time': 0.05915274, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.003458945, 'prompt_tokens_details': None, 'queue_time': 0.280109288, 'total_time': 0.062611685}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_66891002f6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05335-9802-78d1-a635-08c16217f041-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 57, 'total_tokens': 129, 'output_token_details': {'reasoning': 39}})

In [66]:
@tool
def weather_callable(location: str) -> str:
    """Get weather for a location."""
    return f"Demo weather for {location}: 28°C"


In [65]:
# Local Python implementation works independently:
print("Direct Python execution:", weather_callable.invoke("Bangalore"))

Direct Python execution: Demo weather for Bangalore: 28°C


In [68]:
callable_bound_model = groq_llm.bind_tools(
    [weather_callable]
)

In [69]:
ai_message = callable_bound_model.invoke("Use the weather tool for Bangalore.")

[08/30/26 20:45:18] INFO     HTTP Request: POST https://api.groq.com/openai/v1/chat/completions     ]8;id=572957;file://d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=572958;file://d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

In [70]:
ai_message

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the weather tool.', 'tool_calls': [{'id': 'fc_e0f39f89-bb1b-4d08-a690-088d78053964', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'weather_callable'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 126, 'total_tokens': 159, 'completion_time': 0.03394669, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.006079003, 'prompt_tokens_details': None, 'queue_time': 0.28134599, 'total_time': 0.040025693}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c9afb2bdb4', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0533d-34c9-76b3-9b1b-4fbe4bd5451c-0', tool_calls=[{'name': 'weather_callable', 'args': {'location': 'Bangalore'}, 'id': 'fc_e0f39f89-bb1b-4d08-a690-088d78053964', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_to

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the weather tool.', 'tool_calls': [{'id': 'fc_e0f39f89-bb1b-4d08-a690-088d78053964', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'weather_callable'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 126, 'total_tokens': 159, 'completion_time': 0.03394669, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.006079003, 'prompt_tokens_details': None, 'queue_time': 0.28134599, 'total_time': 0.040025693}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c9afb2bdb4', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0533d-34c9-76b3-9b1b-4fbe4bd5451c-0', tool_calls=[{'name': 'weather_callable', 'args': {'location': 'Bangalore'}, 'id': 'fc_e0f39f89-bb1b-4d08-a690-088d78053964', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 126, 'output_tokens': 33, 'total_tokens': 159, 'output_token_details': {'reasoning': 9}})

In [71]:
print("Generated tool calls:")
print(ai_message.tool_calls)

Generated tool calls:
[{'name': 'weather_callable', 'args': {'location': 'Bangalore'}, 'id': 'fc_e0f39f89-bb1b-4d08-a690-088d78053964', 'type': 'tool_call'}]


Generated tool calls:
[{'name': 'weather_callable', 'args': {'location': 'Bangalore'}, 'id': 'fc_e0f39f89-bb1b-4d08-a690-088d78053964', 'type': 'tool_call'}]

In [ ]:
# class GetWeatherSchema(BaseModel):
#     """Get current weather."""
#     location: str = Field(description="City name")

# from langchain_core.utils.function_calling import convert_to_openai_tool

# print("Converted tool schema:")
# print(convert_to_openai_tool(GetWeatherSchema))

In [ ]:

# pydantic_bound_model = groq_llm.bind_tools(
#     [GetWeatherSchema],
#     tool_choice="GetWeatherSchema",
# )

# ai_message = pydantic_bound_model.invoke(
#     "Get the weather for Bangalore."
# )

# print("Generated tool call:")
# print(ai_message.tool_calls)

# print(
# "Important: There is no weather implementation here, "
# "so the returned tool call still needs an executor."
# )

In [ ]:
# class WeatherTypedDict(TypedDict):
#     """Get current weather."""
#     location: str

# print("Converted TypedDict tool schema:")
# print(convert_to_openai_tool(WeatherTypedDict))

In [ ]:
# if groq_llm is not None:
#     typed_dict_model = groq_llm.bind_tools(
#         [WeatherTypedDict],
#         tool_choice="WeatherTypedDict",
#     )

#     ai_message = typed_dict_model.invoke(
#         "Get the weather for New York."
#     )

#     print("Generated tool call:")
#     print(ai_message.tool_calls)

In [ ]:
# raw_weather_tool = {
#     "type": "function",
#     "function": {
#         "name": "get_weather_raw",
#         "description": "Get current weather",
#         "parameters": {
#             "type": "object",
#             "properties": {
#                 "location": {
#                     "type": "string",
#                     "description": "City name",
#                 }
#             },
#             "required": ["location"],
#         },
#     },
# }

# print("Raw schema:")
# print(raw_weather_tool)


In [ ]:
# if groq_llm is not None:
#     raw_bound_model = groq_llm.bind_tools(
#         [raw_weather_tool],
#         tool_choice="get_weather_raw",
#     )

#     ai_message = raw_bound_model.invoke(
#         "Get the weather for Delhi."
#     )

#     print("Generated tool call:")
#     print(ai_message.tool_calls)


##### Toolkit — a collection/factory of related tools

In [ ]:
from langchain_core.tools import BaseToolkit

@tool
def toolkit_add(a: int, b: int) -> int:
    """Add numbers."""
    return a + b

@tool
def toolkit_multiply(a: int, b: int) -> int:
    """Multiply numbers."""
    return a * b

class DemoMathToolkit(BaseToolkit):
    def get_tools(self):
        return [
            toolkit_add,
            toolkit_multiply,
        ]

demo_toolkit = DemoMathToolkit()
toolkit_tools = demo_toolkit.get_tools()

print("Toolkit tools:", [t.name for t in toolkit_tools])
print("Add result:", toolkit_tools[0].invoke({"a": 2, "b": 3}))
print("Multiply result:", toolkit_tools[1].invoke({"a": 4, "b": 5}))

#### Prebuilt integration tool

In [ ]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=3,
    search_depth="basic",
    topic="general",
)

result = tavily_tool.invoke({
    "query": "What is LangGraph?"
})

print(result)

 `bind_tools()` and `ToolNode` are NOT creation methods

 ```text
Create Tool
    ↓
bind_tools()
    ↓
LLM generates tool_calls
    ↓
ToolNode executes the call
    ↓
ToolMessage
    ↓
LLM can continue
```

In [ ]:
from langchain_core.messages import AIMessage
from langgraph.prebuilt import ToolNode

In [ ]:
@tool
def subtract(a: int, b: int) -> int:
    """Subtract b from a."""
    return a - b

In [ ]:
tool_node = ToolNode([subtract])

In [ ]:
tool_node_result = tool_node.invoke({
    "messages": [AIMessage(content="",
                tool_calls=[{
                "name": "subtract",
                "args": {
                    "a": 100,
                    "b": 35,
                },
                "id": "call_subtract_1",
                "type": "tool_call",
            }],
        )
    ]
})

In [ ]:
print("ToolNode result:")
print(tool_node_result)

In [ ]:
print("Tool output:")
print(tool_node_result["messages"][-1].content)

#### Create an MCP tool and load it into LangChain

```bash
uv pip install -U mcp langchain-mcp-adapters
```

In [53]:
from pathlib import Path
import sys

In [52]:
from mcp.server.fastmcp import FastMCP

In [54]:
mcp = FastMCP("Math")

In [55]:
@mcp.tool()
def add_mcp(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [57]:
mcp.run(transport="stdio")

RuntimeError: Already running asyncio in this thread

In [58]:
mcp_server_path = Path("demo_math_mcp_server.py")
mcp_server_path.write_text(mcp_server_code, encoding="utf-8")

print("Created MCP server:", mcp_server_path.resolve())

NameError: name 'mcp_server_code' is not defined

In [59]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [ ]:
mcp_client = MultiServerMCPClient({
    "math": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(mcp_server_path.resolve())],
    }
})

In [ ]:
mcp_tools = await mcp_client.get_tools()

In [ ]:
print("Loaded MCP tools:", [t.name for t in mcp_tools])

In [ ]:
add_mcp_tool = next(
        t for t in mcp_tools
        if t.name == "add_mcp"
    )


In [ ]:
print(
    "MCP execution result:",
    await add_mcp_tool.ainvoke({
        "a": 10,
        "b": 25,
    })
    )

In [ ]:
server_1 = Path("demo_mcp_add_server.py")
server_2 = Path("demo_mcp_multiply_server.py")

server_1.write_text(r'''
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("AddServer")

@mcp.tool()
def remote_add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

if __name__ == "__main__":
    mcp.run(transport="stdio")
''', encoding="utf-8")

server_2.write_text(r'''
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("MultiplyServer")

@mcp.tool()
def remote_multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")
''', encoding="utf-8")

print("Created two MCP server files.")

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

multi_client = MultiServerMCPClient({
    "addition": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(server_1.resolve())],
    },
    "multiplication": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(server_2.resolve())],
    },
})

multi_tools = await multi_client.get_tools()

print("Tools loaded from multiple MCP servers:")
print([t.name for t in multi_tools])

tool_map = {
    t.name: t
    for t in multi_tools
}

print(
    "remote_add:",
    await tool_map["remote_add"].ainvoke({
        "a": 5,
        "b": 6,
    })
)

print(
    "remote_multiply:",
    await tool_map["remote_multiply"].ainvoke({
        "a": 5,
        "b": 6,
    })
)

In [ ]:
@tool
def langchain_add_for_mcp(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [ ]:
try:
    from langchain_mcp_adapters.tools import to_fastmcp

    fastmcp_tool = to_fastmcp(
        langchain_add_for_mcp
    )

    print("Converted FastMCP tool:")
    print(fastmcp_tool)

except ImportError:
    print("Install langchain-mcp-adapters to run this section.")

1. First understand what a tool actually is

A tool is basically a capability with:

Tool
 ├── Name
 ├── Description
 ├── Input Schema
 └── Actual Functionality

Example:

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

The LLM sees something conceptually like:

Name:
add

Description:
Add two numbers.

Arguments:
a: integer
b: integer

Then the LLM can decide:

User:
"What is 20 + 30?"

      ↓

LLM:
I should call add.

      ↓

Tool Call:
add(a=20, b=30)

      ↓

Application executes function

      ↓

50

The important part:

The LLM usually chooses the tool and arguments; Python/application code actually executes it.

2. Your most important distinction: Node vs Tool

Your notebook currently says:

Node = graph decides when to execute the function.
Tool = LLM decides when to execute the function.

For teaching beginners, this is a very useful mental model, but technically I would slightly improve it.

Better definition:

Node = a workflow/orchestration step.
Tool = a capability exposed through a name, description and input schema, usually so an LLM/agent can invoke it.

Why am I changing it slightly?

Because a tool can also be invoked manually:

tool.invoke(...)

and a node can contain an LLM that dynamically decides things.

So this is more technically accurate:

Node
=
workflow step

Tool
=
callable capability

3. Suppose I already have a Python function. Why create a tool?

This is probably the most important question.

Suppose:

def get_weather(city: str):
    return weather_api(city)

You could absolutely make it a node:

def weather_node(state):
    result = get_weather(
        state["city"]
    )

    return {
        "weather": result
    }

Then:

builder.add_node(
    "weather",
    weather_node
)

Graph controls execution:

START
  ↓
Weather Node
  ↓
Final

This is perfect if your workflow already knows:

Weather must execute here.

But suppose the user can ask:

"What is the weather in Bangalore?"

"What is 20 + 30?"

"Search my company policy."

"Tell me what machine learning is."

Now your agent has:

Weather
Calculator
Retriever
Web Search
Database
Email

If everything is only nodes, you need explicit routing:

             Router
          /    |     \
         /     |      \
Weather Node Calculator RAG Node

As capabilities increase:

50 tools
100 tools

your graph can become complicated.

Instead:

tools = [
    weather,
    calculator,
    retriever,
    web_search
]

model_with_tools = model.bind_tools(tools)

Now:

User
 ↓
LLM
 ↓
Which capability do I need?
 ↓
Tool Call
 ↓
Execute Tool

This is the main benefit.

4. Node = deterministic orchestration

Suppose this must happen for every answer:

Generate Answer
      ↓
Safety Validation
      ↓
Audit Logging
      ↓
Final

These should normally be nodes.

Why?

You do NOT want:

LLM:
"Should I run safety validation?"

Maybe yes
Maybe no

No.

Graph should guarantee it:

LLM Node
   ↓
Validator Node
   ↓
Audit Node
   ↓
END

So use nodes for:

Mandatory processing
Deterministic workflows
Validation
Guardrails
Logging
State transformation
Routing
Retry logic
Finalization

5. Tool = dynamic capability

Use tools when:

The model should decide whether the capability is needed.

For example:

Calculator
Web search
Database query
Weather
Stock API
Email
Calendar
Retriever
GitHub
Slack
File operations

Example:

User:
"Tell me today's Tesla price."

LLM
 ↓
Needs current information
 ↓
Stock Tool
 ↓
Result
 ↓
LLM

But:

User:
"Explain overfitting."

LLM
 ↓
No tool required
 ↓
Answer directly

That dynamic decision is why tools exist.

6. The ideal architecture uses BOTH

This is what I would teach:

                    START
                      ↓
                  Agent Node
                      ↓
                 LLM + Tools
                      ↓
          ┌───────────┼───────────┐
          ↓           ↓           ↓
      Calculator     Web       Retriever
        Tool         Tool        Tool
          └───────────┼───────────┘
                      ↓
                  Agent Node
                      ↓
                Validator Node
                      ↓
                 Final Node
                      ↓
                     END

Here:

Agent
Validator
Final

are nodes.

And:

Calculator
Web
Retriever

are tools.

This is a clean design.

Now let's analyze every method in YOUR CODE

| Priority | Method                           | When                                 |
| -------- | -------------------------------- | ------------------------------------ |
| ⭐⭐⭐⭐⭐    | `@tool`                          | Default for normal custom functions  |
| ⭐⭐⭐⭐⭐    | `@tool + Pydantic`               | Complex/validated business input     |
| ⭐⭐⭐⭐     | async `@tool`                    | API/DB/network tools                 |
| ⭐⭐⭐⭐     | `create_retriever_tool()`        | Agentic RAG                          |
| ⭐⭐⭐⭐     | Prebuilt integrations            | Tavily, DB, APIs, etc.               |
| ⭐⭐⭐⭐     | MCP                              | External/shared enterprise tools     |
| ⭐⭐⭐      | `ToolRuntime`                    | Tool needs graph state/context/store |
| ⭐⭐⭐      | Runnable `.as_tool()`            | Existing chain/Runnable becomes tool |
| ⭐⭐       | `StructuredTool.from_function()` | Explicit structured construction     |
| ⭐⭐       | `BaseTool`                       | Advanced custom integration          |
| ⭐        | `Tool(...)` / `.from_function()` | Simple explicit wrappers             |
| ⭐        | Raw JSON schemas                 | Provider-level/low-level work        |


My decision tree

Use this in your class:

I have a normal Python function
        |
        v
      @tool
        |
        +--------------------------+
        |                          |
Simple inputs?               Complex inputs?
        |                          |
       YES                         YES
        |                          |
      @tool              @tool + Pydantic

Then:

Does it call network/API/DB?
        |
       YES
        |
        v
   async @tool

Then:

Is it a Retriever?
        |
       YES
        |
        v
create_retriever_tool()

Then:

Already have an LCEL/Runnable?
        |
       YES
        |
        v
   runnable.as_tool()

Then:

Need graph state/context/store?
        |
       YES
        |
        v
     ToolRuntime

Then:

Need very deep customization?
        |
       YES
        |
        v
     BaseTool

Then:

Tool exists remotely / shared
across many applications?
        |
       YES
        |
        v
        MCP

Node or Tool? Final decision tree

This is the most useful one.

Ask:

Should my workflow guarantee WHEN this function runs?

If yes:

NODE

Example:

Validate every response.

Graph:

LLM
 ↓
Validator
 ↓
END

Ask:

Should the LLM decide WHETHER it needs this capability?

If yes:

TOOL

Example:

Maybe use calculator.
Maybe use search.
Maybe don't use anything.

Flow:

LLM
 ↓
Tool needed?
 /       \
Yes       No
 ↓         ↓
Tool     Answer